In [1]:
# ===================================
# Stopwords
# ===================================
def load_stopwords(path):
    with open(path, encoding="utf-8") as f:
        return set(line.strip() for line in f)


# ===================================
# Persian NLP
# ===================================
from hazm import Normalizer, word_tokenize, Stemmer
import re
normalizer = Normalizer()
stemmer = Stemmer()

def preprocess_text(text, stopwords):
    text = normalizer.normalize(text)
    text = re.sub(r"[^\u0600-\u06FF\s]", " ", text)
    tokens = word_tokenize(text)
    return [stemmer.stem(token) for token in tokens if token not in stopwords and len(token) > 1]


# ===================================
# Load Hamshahri Corpus
# ===================================
import os

def load_hamshahri_corpus(base_path):
    documents = {}
    doc_id_to_path = {}
    for year in os.listdir(base_path):
        year_path = os.path.join(base_path, year)
        if not os.path.isdir(year_path):
            continue
        for file in os.listdir(year_path):
            if file.endswith(".ham"):
                path = os.path.join(year_path, file)
                with open(path, encoding="utf-8", errors="ignore") as f:
                    documents[file.replace(".ham", "")] = f.read()
                doc_id_to_path[file.replace(".ham", "")] = path
    return documents, doc_id_to_path


# ===================================
# Load Queries
# ===================================
def load_queries(path):
    queries = {}
    for file in os.listdir(path):
        if file.endswith(".q"):
            with open(os.path.join(path, file), encoding="utf-8") as f:
                queries[file.replace(".q", "")] = f.read().strip()
    return queries


# ===================================
# Load Judgments
# ===================================
from collections import defaultdict

def load_judgments(path):
    relevance = defaultdict(set)
    with open(path, encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 2:
                relevance[parts[0]].add(parts[1])
            elif len(parts) == 3 and int(parts[2]) > 0:
                relevance[parts[0]].add(parts[1])
    return relevance


# ===================================
# TF-IDF (MANUAL)
# ===================================
import math
import numpy as np

def build_tfidf(processed_docs):
    doc_ids = list(processed_docs.keys())
    vocab = {}
    for tokens in processed_docs.values():
        for t in tokens:
            if t not in vocab:
                vocab[t] = len(vocab)
    N = len(doc_ids)
    V = len(vocab)
    # DF
    df = [0] * V
    for tokens in processed_docs.values():
        for t in set(tokens):
            df[vocab[t]] += 1
    # IDF
    idf = [math.log(N / (df_i + 1)) for df_i in df]
    # TF-IDF matrix
    tfidf = np.zeros((N, V))
    for d, doc_id in enumerate(doc_ids):
        tokens = processed_docs[doc_id]
        counts = {}
        for t in tokens:
            counts[t] = counts.get(t, 0) + 1
        for t, c in counts.items():
            idx = vocab[t]
            tfidf[d][idx] = (c / len(tokens)) * idf[idx]
    return tfidf, vocab, idf, doc_ids


# ===================================
# Cosine Similarity (Sparse Query)
# ===================================
def vector_norm(vec):
    return math.sqrt(sum(v*v for v in vec))

def cosine_similarity_sparse(q_sparse, doc_vec, q_norm, doc_norm):
    dot = 0.0
    for idx, q_val in q_sparse.items():
        dot += q_val * doc_vec[idx]
    return dot / (q_norm * doc_norm + 1e-8)


# ===================================
# Retrieval
# ===================================
def retrieve(query, vocab, idf, tfidf_matrix, doc_ids,
             preprocess_func, stopwords, doc_norms, k):
    tokens = preprocess_func(query, stopwords)
    q_sparse = {}
    counts = {}
    for t in tokens:
        if t in vocab:
            counts[t] = counts.get(t, 0) + 1
    for t, c in counts.items():
        idx = vocab[t]
        q_sparse[idx] = (c / len(tokens)) * idf[idx]
    q_norm = math.sqrt(sum(v*v for v in q_sparse.values()))
    scores = []
    for i, doc_vec in enumerate(tfidf_matrix):
        sim = cosine_similarity_sparse(q_sparse, doc_vec, q_norm, doc_norms[i])
        scores.append((doc_ids[i], sim))
    scores.sort(key=lambda x: x[1], reverse=True)
    return [doc for doc, _ in scores[:k]]


# ===================================
# Metrics
# ===================================
def precision(r, rel): return len(set(r) & rel) / len(r) if r else 0
def recall(r, rel): return len(set(r) & rel) / len(rel) if rel else 0
def f1(p, r): return (2*p*r)/(p+r) if (p+r) else 0

def average_precision(ranked, relevant):
    score = hits = 0
    for i, d in enumerate(ranked):
        if d in relevant:
            hits += 1
            score += hits / (i+1)
    return score / len(relevant) if relevant else 0


# ===================================
# Paths
# ===================================
BASE = "HamshahriData"
STOPWORDS_PATH = BASE + "/persian_stopwords.txt"
CORPUS_PATH = BASE + "/HamshahriCorpus"
QUERY_PATH = BASE + "/Queries"
JUDGE_PATH = BASE + "/RelativeAssesemnt/judgements.txt"


# ===================================
# Load Data
# ===================================
stopwords = load_stopwords(STOPWORDS_PATH)
documents, doc_id_to_path = load_hamshahri_corpus(CORPUS_PATH)
queries = load_queries(QUERY_PATH)
judgments = load_judgments(JUDGE_PATH)

processed_docs = {d: preprocess_text(t, stopwords) for d, t in documents.items()}

tfidf_matrix, vocab, idf, doc_ids = build_tfidf(processed_docs)
doc_norms = [vector_norm(vec) for vec in tfidf_matrix]


# ===================================
# Evaluation
# ===================================
limit = 10
P, R, F, APs = [], [], [], []

print("\n=== System Evaluation ===\n")

for q_id, q_text in queries.items():
    retrieved = retrieve(
        q_text, vocab, idf, tfidf_matrix,
        doc_ids, preprocess_text, stopwords,
        doc_norms, k=len(doc_ids)
    )
    rel = judgments.get(q_id, set())
    p = precision(retrieved[:limit], rel)
    r = recall(retrieved[:limit], rel)
    f = f1(p, r)
    ap = average_precision(retrieved, rel)
    P.append(p); R.append(r); F.append(f); APs.append(ap)

print(f"Mean Precision@{limit} = {sum(P)/len(P):.4f}")
print(f"Mean Recall@{limit} = {sum(R)/len(R):.4f}")
print(f"Mean F1-score@{limit} = {sum(F)/len(F):.4f}")
print(f"MAP = {sum(APs)/len(APs):.4f}\n")


# ===================================
# Display Sample Results
# ===================================
top_k = 5
sample_queries = list(queries.items())[:50]

for q_id, q_text in sample_queries:
    tokens = preprocess_text(q_text, stopwords)
    q_sparse = {}
    counts = {}
    for t in tokens:
        if t in vocab:
            counts[t] = counts.get(t, 0) + 1
    for t, c in counts.items():
        q_sparse[vocab[t]] = (c / len(tokens)) * idf[vocab[t]]
    q_norm = math.sqrt(sum(v*v for v in q_sparse.values()))

    scores = []
    for i, doc_vec in enumerate(tfidf_matrix):
        sim = cosine_similarity_sparse(q_sparse, doc_vec, q_norm, doc_norms[i])
        scores.append((doc_ids[i], sim))
    scores.sort(key=lambda x: x[1], reverse=True)

    print(f"Query {q_id}: {q_text}")
    print(f"Top-{top_k} Retrieved Documents with Similarity and Year:")

    for rank, (doc_id, score) in enumerate(scores[:top_k], start=1):
        year = os.path.basename(os.path.dirname(doc_id_to_path[doc_id]))
        print(f"{rank}. {doc_id} - Similarity: {score*100:.2f}% - Year: {year}")
    print("-" * 50)



=== System Evaluation ===

Mean Precision@10 = 0.7680
Mean Recall@10 = 0.3227
Mean F1-score@10 = 0.3788
MAP = 0.7441

Query 1: بازسازي شهر زلزله زده بم
Top-5 Retrieved Documents with Similarity and Year:
1. HAM2-830506-029 - Similarity: 56.71% - Year: 2004
2. HAM2-821017-065 - Similarity: 53.92% - Year: 2004
3. HAM2-831005-128 - Similarity: 48.31% - Year: 2004
4. HAM2-821007-088 - Similarity: 45.63% - Year: 2003
5. HAM2-840818-157 - Similarity: 44.20% - Year: 2005
--------------------------------------------------
Query 10: بهره برداري از تونل رسالت
Top-5 Retrieved Documents with Similarity and Year:
1. HAM2-840705-124 - Similarity: 70.67% - Year: 2005
2. HAM2-850427-078 - Similarity: 68.71% - Year: 2006
3. HAM2-850413-063 - Similarity: 68.34% - Year: 2006
4. HAM2-850323-081 - Similarity: 65.49% - Year: 2006
5. HAM2-850328-121 - Similarity: 64.69% - Year: 2006
--------------------------------------------------
Query 11: نرخ بيكاري در ايران
Top-5 Retrieved Documents with Similarity and